# Imports and settings


In [1]:
import pandas as pd
import numpy as np
import torch
from transformers import BertTokenizer, BertForMaskedLM
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    average_precision_score,
    roc_auc_score,
)
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.05)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


In [2]:
# Constant definitions
BATCH_SIZE = 32  # Batch size for gpu processing
FRACTION_SAMPLE = 0.1  # Use a fraction of the data for faster experimentation
RANDOM_STATE = 42
DATA_PATH = "./data/preprocessed/steam_reviews_preprocessed.csv"
OUTPUT_PATH = "outputs/bert_baseline/"

In [3]:
# Load BERT model and tokenizer
model_name = "bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertForMaskedLM.from_pretrained(model_name).to(device)
model.eval()

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertForMaskedLM(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_a

In [4]:
# prompts and word lists for humor prediction

prompts_dezimal = [
    "On a scale from 0.0 to 1.0 this review is 0.[MASK] funny.",
    "On a scale from 0 to 1 this review is 0.[MASK] funny.",
]
cat_dezimal = ["0", "1", "2", "3", "4", "5", "6", "7", "8", "9"]

prompts_words = [
    "Overall, the humor is [MASK].",
    "Overall the humor of this review is [MASK].",
    "The humor in this review is [MASK].",
]
cat_words = ["serious", "witty", "amusing", "hilarious", "hysterical"]

In [5]:
# Check prompt token lengths
for prompt in prompts_dezimal + prompts_words:
    toks = tokenizer.tokenize(prompt)
    print(f"Prompt: '{prompt}'  Tokens: {toks}  Length: {len(toks)}")


# check tokenization of category words is single token
for w in cat_dezimal + cat_words:
    toks = tokenizer.tokenize(w)
    print(f"Word: '{w}'  Tokens: {toks}  Length: {len(toks)}")
    if len(toks) > 1:
        print(f"  WARNING: '{w}' tokenizes into multiple tokens: {toks}")

Prompt: 'On a scale from 0.0 to 1.0 this review is 0.[MASK] funny.'  Tokens: ['on', 'a', 'scale', 'from', '0', '.', '0', 'to', '1', '.', '0', 'this', 'review', 'is', '0', '.', '[MASK]', 'funny', '.']  Length: 19
Prompt: 'On a scale from 0 to 1 this review is 0.[MASK] funny.'  Tokens: ['on', 'a', 'scale', 'from', '0', 'to', '1', 'this', 'review', 'is', '0', '.', '[MASK]', 'funny', '.']  Length: 15
Prompt: 'Overall, the humor is [MASK].'  Tokens: ['overall', ',', 'the', 'humor', 'is', '[MASK]', '.']  Length: 7
Prompt: 'Overall the humor of this review is [MASK].'  Tokens: ['overall', 'the', 'humor', 'of', 'this', 'review', 'is', '[MASK]', '.']  Length: 9
Prompt: 'The humor in this review is [MASK].'  Tokens: ['the', 'humor', 'in', 'this', 'review', 'is', '[MASK]', '.']  Length: 8
Word: '0'  Tokens: ['0']  Length: 1
Word: '1'  Tokens: ['1']  Length: 1
Word: '2'  Tokens: ['2']  Length: 1
Word: '3'  Tokens: ['3']  Length: 1
Word: '4'  Tokens: ['4']  Length: 1
Word: '5'  Tokens: ['5']  Lengt

# Dataset

Loading, preprocessing and sampling of the dataset


In [6]:
# Load data
df = pd.read_csv(DATA_PATH)

In [7]:
# Drop rows with NaN in review_text_cleaned (not sure how they are still here)
before_count = len(df)
dropped_ids_rev = df.loc[df["review_text_cleaned"].isna(), "review_id"].tolist()
dropped_ids_rev_info = df.loc[
    df["review_info_text_cleaned"].isna(), "review_id"
].tolist()

df = df.dropna(subset=["review_text_cleaned"])
df = df.dropna(subset=["review_info_text_cleaned"])
after_count = len(df)
print(f"Dropped {before_count - after_count} rows. New size: {after_count}")
print(f"Dropped review_ids (count={len(dropped_ids_rev)}): {dropped_ids_rev}")
print(f"Dropped review_ids (count={len(dropped_ids_rev_info)}): {dropped_ids_rev_info}")

Dropped 0 rows. New size: 719550
Dropped review_ids (count=0): []
Dropped review_ids (count=0): []


In [8]:
# # analyse how many reviews we would have to drop for just the review text when combining with prompt to not exceed 512 tokens
tqdm.pandas(desc="Calculating review lengths")
df["review_char_len"] = df["review_text_cleaned"].astype(str).str.len()
df["review_token_len"] = (
    df["review_text_cleaned"]
    .astype(str)
    .progress_apply(lambda x: len(tokenizer.encode(x, add_special_tokens=False)))
)

prompt_stats = []
for prompt in prompts_dezimal + prompts_words:
    prompt_tokens = tokenizer.tokenize(prompt, add_special_tokens=False)
    max_allowed_len = 512 - len(prompt_tokens) - 2  # Account for [CLS] and [SEP]
    exceed_count = (df["review_token_len"] > max_allowed_len).sum()
    prompt_stats.append(
        {
            "prompt": prompt,
            "prompt_token_len": len(prompt_tokens),
            "max_review_tokens": max_allowed_len,
            "reviews_exceeding_limit": exceed_count,
            "exceeding_percentage": exceed_count / len(df) * 100,
        }
    )

Calculating review lengths:   0%|          | 0/719550 [00:00<?, ?it/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (738 > 512). Running this sequence through the model will result in indexing errors


In [9]:
prompt_stats_df = pd.DataFrame(prompt_stats).sort_values(
    "prompt_token_len", ascending=False
)

print("=== Review length summary ===")
print(
    df[["review_char_len", "review_token_len"]]
    .describe(percentiles=[0.5, 0.9, 0.95, 0.99])
    .T
)

print("\n=== Prompt token lengths and truncation impact ===")
print(prompt_stats_df.to_string(index=False))

=== Review length summary ===
                     count        mean         std  min   50%    90%     95%  \
review_char_len   719550.0  239.978399  540.536463  1.0  73.0  573.0  1009.0   
review_token_len  719550.0   55.833874  127.063653  0.0  18.0  131.0   231.0   

                     99%     max  
review_char_len   2638.0  8000.0  
review_token_len   605.0  7988.0  

=== Prompt token lengths and truncation impact ===
                                                   prompt  prompt_token_len  max_review_tokens  reviews_exceeding_limit  exceeding_percentage
On a scale from 0.0 to 1.0 this review is 0.[MASK] funny.                19                491                    10913              1.516642
    On a scale from 0 to 1 this review is 0.[MASK] funny.                15                495                    10627              1.476895
              Overall the humor of this review is [MASK].                 9                501                    10394              1.444514
    

In [10]:
# check max lengths of app short descriptions, app categories, and app genres to see if they might also cause truncation issues
# Check max lengths of app metadata fields
print("=== App metadata field lengths ===")
print(
    f"app_short_description max char length: {df['app_short_description'].astype(str).str.len().max()}"
)
print(
    f"app_categories max char length: {df['app_categories'].apply(lambda x: len(', '.join(eval(x))) if isinstance(eval(x), list) else 0).max()}"
)
print(
    f"app_genres max char length: {df['app_genres'].apply(lambda x: len(', '.join(eval(x))) if isinstance(eval(x), list) else 0).max()}"
)

=== App metadata field lengths ===
app_short_description max char length: 303
app_categories max char length: 420
app_genres max char length: 59


In [11]:
# Convert app_categories and app_genres to token lengths
tqdm.pandas(desc="Calculating app metadata token lengths")

df["app_short_description_token_len"] = (
    df["app_short_description"]
    .astype(str)
    .progress_apply(lambda x: len(tokenizer.encode(x, add_special_tokens=False)))
)

df["app_categories_token_len"] = df["app_categories"].progress_apply(
    lambda x: len(
        tokenizer.encode(
            ", ".join(eval(x)) if isinstance(eval(x), list) else "",
            add_special_tokens=False,
        )
    )
)

df["app_genres_token_len"] = df["app_genres"].progress_apply(
    lambda x: len(
        tokenizer.encode(
            ", ".join(eval(x)) if isinstance(eval(x), list) else "",
            add_special_tokens=False,
        )
    )
)

print("\n=== App metadata token length statistics ===")
print(
    df[
        [
            "app_short_description_token_len",
            "app_categories_token_len",
            "app_genres_token_len",
        ]
    ]
    .describe()
    .T
)

Calculating app metadata token lengths:   0%|          | 0/719550 [00:00<?, ?it/s]

Calculating app metadata token lengths:   0%|          | 0/719550 [00:00<?, ?it/s]

Calculating app metadata token lengths:   0%|          | 0/719550 [00:00<?, ?it/s]


=== App metadata token length statistics ===
                                    count       mean        std   min   25%  \
app_short_description_token_len  719550.0  43.269172  11.798120  16.0  31.0   
app_categories_token_len         719550.0  52.749213  18.096876  24.0  39.0   
app_genres_token_len             719550.0   4.315275   2.864326   1.0   3.0   

                                  50%   75%   max  
app_short_description_token_len  47.0  52.0  59.0  
app_categories_token_len         48.0  61.0  88.0  
app_genres_token_len              3.0   7.0  12.0  


In [12]:
# Analysis of length of reviews
tqdm.pandas(desc="Calculating review and info lengths")
df["review_info_char_len"] = df["review_info_text_cleaned"].astype(str).str.len()
df["review_info_token_len"] = (
    df["review_info_text_cleaned"]
    .astype(str)
    .progress_apply(lambda x: len(tokenizer.encode(x, add_special_tokens=False)))
)

prompt_stats = []
for prompt in prompts_dezimal + prompts_words:
    prompt_tokens = tokenizer.tokenize(prompt, add_special_tokens=False)
    max_allowed_len = 512 - len(prompt_tokens) - 2  # Account for [CLS] and [SEP]
    exceed_count = (df["review_info_token_len"] > max_allowed_len).sum()
    prompt_stats.append(
        {
            "prompt": prompt,
            "prompt_token_len": len(prompt_tokens),
            "max_review_tokens": max_allowed_len,
            "reviews_exceeding_limit": exceed_count,
            "exceeding_percentage": exceed_count / len(df) * 100,
        }
    )

Calculating review and info lengths:   0%|          | 0/719550 [00:00<?, ?it/s]

In [13]:
prompt_stats_df_info = pd.DataFrame(prompt_stats).sort_values(
    "prompt_token_len", ascending=False
)

print("=== Review length summary ===")
print(
    df[["review_info_char_len", "review_info_token_len"]]
    .describe(percentiles=[0.5, 0.9, 0.95, 0.99])
    .T
)

print("\n=== Prompt token lengths and truncation impact ===")
print(prompt_stats_df_info.to_string(index=False))

=== Review length summary ===
                          count        mean         std    min    50%     90%  \
review_info_char_len   719550.0  797.905574  545.398458  394.0  660.0  1136.0   
review_info_token_len  719550.0  174.668630  127.515765   83.0  143.0   250.0   

                          95%     99%     max  
review_info_char_len   1560.0  3180.0  8780.0  
review_info_token_len   348.0   721.0  8107.0  

=== Prompt token lengths and truncation impact ===
                                                   prompt  prompt_token_len  max_review_tokens  reviews_exceeding_limit  exceeding_percentage
On a scale from 0.0 to 1.0 this review is 0.[MASK] funny.                19                491                    17920              2.490445
    On a scale from 0 to 1 this review is 0.[MASK] funny.                15                495                    17650              2.452922
              Overall the humor of this review is [MASK].                 9                501          

In [14]:
# Drop reviews + info texts that exceed token length when combined with longest prompt (19 Tokens) 512 - 19 -2
max_tokens_for_dezimal = 490
df = df[df["review_info_token_len"] <= max_tokens_for_dezimal].reset_index(drop=True)
print(f"Dropped reviews exceeding token limit. Remaining reviews: {len(df):,}")

Dropped reviews exceeding token limit. Remaining reviews: 701,570


## Create balanced dataset that has equal parts funny and not funny reviews

so we don't have to wait 5 hours for the model to predict only reviews with 0 votes


In [15]:
print(f"Number of funny reviews: {len(df[df["label_is_funny_binary"] == 1])}")
print(f"Number of not funny reviews: {len(df[df["label_is_funny_binary"] == 0])}")
print(f"Max number of up_votes: {df["review_votes_funny"].max()}")

Number of funny reviews: 48799
Number of not funny reviews: 652771
Max number of up_votes: 2089


In [16]:
# Create a kind of balanced dataset: Same amount of funny and not-funny reviews for better evaluation of the model's ability to distinguish humor
funny_df = df[df["label_is_funny_binary"] == 1].sample(
    frac=FRACTION_SAMPLE, random_state=RANDOM_STATE
)
not_funny_df = df[df["label_is_funny_binary"] == 0].sample(
    n=len(funny_df), random_state=RANDOM_STATE
)
sample = (
    pd.concat([funny_df, not_funny_df])
    .sample(frac=1, random_state=RANDOM_STATE)
    .reset_index(drop=True)
)

print(
    f"Sample size: {len(sample):,}  |  Funny: {sample['label_is_funny_binary'].sum():,}  |  Not Funny: {(sample['label_is_funny_binary'] == 0).sum():,}"
)

Sample size: 9,760  |  Funny: 4,880  |  Not Funny: 4,880


# Masked Language Modeling (MLM)

Using the Prompts we defined at the start we are inspecting the words that bert-base-uncased would predict for the [MASK] token.


In [17]:
# Helper functions to predict humor for single reviews and spot check representative reviews with detailed output
def predict_single_review(
    text,
    prompt,
    tokenizer,
    model,
    device,
    words,
    top_k=10,
    max_length=512,
):
    """Run a masked-LM prediction for a single review and print detailed results."""
    # just to be sure (even though  we already did this) calculate max review_tokens length and truncate
    target_ids = {w: tokenizer.convert_tokens_to_ids(w) for w in words}
    mask_id = tokenizer.mask_token_id
    prompt_token_count = len(tokenizer.encode(prompt, add_special_tokens=False))
    max_review_tokens = max_length - prompt_token_count - 2
    review_ids = tokenizer.encode(text, add_special_tokens=False)[:max_review_tokens]
    review_text = tokenizer.decode(review_ids, skip_special_tokens=True)
    combined = f"{review_text} {prompt}"

    inputs = tokenizer(
        combined, return_tensors="pt", truncation=True, max_length=max_length
    ).to(device)

    with torch.no_grad():
        logits = model(**inputs).logits.squeeze(0)

    mask_pos = (
        (inputs["input_ids"].squeeze(0) == mask_id).nonzero(as_tuple=False).item()
    )
    probs = torch.softmax(logits[mask_pos], dim=-1)

    top_probs, top_indices = probs.topk(top_k)
    top_tokens = tokenizer.convert_ids_to_tokens(top_indices.cpu().tolist())
    top_k_list = list(zip(top_tokens, [round(p, 6) for p in top_probs.cpu().tolist()]))

    target_scores = {w: round(probs[tid].item(), 6) for w, tid in target_ids.items()}

    return {
        "top_k": top_k_list,
        "target_scores": target_scores,
    }

#Check three reviews explicitly. Highest funny votes, median funny votes, zero funny votes
def spot_check_reviews(sample, column, prompt, tokenizer, model, device, words):
    """Pick three representative reviews and run predictions on each."""

    # Filter to keep only single-token words
    discarded_words = [
        w for w in words if len(tokenizer.encode(w, add_special_tokens=False)) != 1
    ]

    if discarded_words:
        print(f"Discarded multi-token words: {discarded_words}")

    words = [
        w for w in words if len(tokenizer.encode(w, add_special_tokens=False)) == 1
    ]

    # 1. Highest funny votes
    row_max = sample.loc[sample["review_votes_funny"].idxmax()]

    # 2. Median funny votes (closest to median)
    median_votes = sample["review_votes_funny"].median()
    row_med = sample.iloc[
        (sample["review_votes_funny"] - median_votes).abs().argsort().iloc[0]
    ]

    # 3. Zero funny votes (random pick)
    zero_df = sample[sample["review_votes_funny"] == 0]
    row_zero = zero_df.sample(n=1, random_state=RANDOM_STATE).iloc[0]

    picks = [
        ("HIGHEST funny votes", row_max),
        ("MEDIAN funny votes", row_med),
        ("ZERO funny votes", row_zero),
    ]

    print(f'Prompt: "{prompt}"\n')
    for label, row in picks:
        result = predict_single_review(
            row[column], prompt, tokenizer, model, device, words
        )
        print(f"=== {label} ===")
        print(f"  Review ID : {row['review_id']}")
        print(f"  Game      : {row['app_name']}")
        print(
            f"  Funny Votes: {row['review_votes_funny']}  |  Label: {'Funny' if row['label_is_funny_binary'] == 1 else 'Not Funny'}"
        )
        print(f"  Text      : {row[column]}")
        print(f"  Top-{len(result['top_k'])}: {result['top_k']}")

        funny_str = ", ".join(
            f"{w}: {result['target_scores'][w]}"
            for w in sorted(
                words, key=lambda w: result["target_scores"][w], reverse=True
            )
        )
        print(f"  Words     : {{{funny_str}}}")

In [26]:
def get_mask_predictions(
    sample,
    column,
    prompt,
    tokenizer,
    model,
    device,
    words,
    top_k=10,
    max_length=512,
    batch_size=BATCH_SIZE,
):
    """
    For each review, append `prompt` (which must contain [MASK]),
    then return the top-k predicted words at the [MASK] position
    and the probabilities for every word in TARGET_WORDS.

    Processes texts in batches for faster GPU execution.

    Returns a list of dicts, one per text:
        {
            "top_k": [(word, prob), ...],          # top-k predictions
            "target_scores": {word: prob, ...},     # scores for TARGET_WORDS
            "funny_sum": float,                     # sum of funny word probs
            "not_funny_sum": float,                 # sum of not-funny word probs
        }
    """

    texts = sample[column].tolist()

    # Filter to keep only single-token words
    discarded_words = [
        w for w in words if len(tokenizer.encode(w, add_special_tokens=False)) != 1
    ]

    if discarded_words:
        print(f"Discarded multi-token words: {discarded_words}")

    words = [
        w for w in words if len(tokenizer.encode(w, add_special_tokens=False)) == 1
    ]

    target_ids = {w: tokenizer.convert_tokens_to_ids(w) for w in words}
    target_id_tensor = torch.tensor([target_ids[w] for w in words], device=device)
    mask_id = tokenizer.mask_token_id

    # Reserve tokens for: [CLS] + prompt tokens + [SEP]
    prompt_token_count = len(tokenizer.encode(prompt, add_special_tokens=False))
    max_review_tokens = max_length - prompt_token_count - 2

    # Pre-truncate all reviews so the prompt (with [MASK]) is never cut off
    combined_texts = []
    for text in texts:
        review_ids = tokenizer.encode(text, add_special_tokens=False)[
            :max_review_tokens
        ]
        review_text = tokenizer.decode(review_ids, skip_special_tokens=True)
        combined_texts.append(f"{review_text} {prompt}")

    results = []
    num_batches = (len(combined_texts) + batch_size - 1) // batch_size

    for batch_idx in tqdm(range(num_batches), desc="Predicting (batched)"):
        start = batch_idx * batch_size
        end = min(start + batch_size, len(combined_texts))
        batch_texts = combined_texts[start:end]

        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            truncation=True,
            max_length=max_length,
            padding=True,
        ).to(device)

        with torch.no_grad():
            logits = model(**inputs).logits  # (batch, seq_len, vocab)

        input_ids = inputs["input_ids"]  # (batch, seq_len)

        # Find [MASK] positions for each item in the batch
        mask_positions = (input_ids == mask_id).nonzero(
            as_tuple=False
        )  # (N, 2) -> [batch_idx, seq_pos]

        for i in range(len(batch_texts)):
            # Get the mask position for this item
            item_masks = mask_positions[mask_positions[:, 0] == i]
            mask_pos = item_masks[0, 1].item()

            probs = torch.softmax(logits[i, mask_pos], dim=-1)

            # Top-k predictions
            top_probs, top_indices = probs.topk(top_k)
            top_tokens = tokenizer.convert_ids_to_tokens(top_indices.cpu().tolist())
            top_k_list = list(
                zip(top_tokens, [round(p, 6) for p in top_probs.cpu().tolist()])
            )

            # Target-word scores (vectorized lookup)
            target_probs = probs[target_id_tensor].cpu().tolist()
            target_scores = {w: round(p, 6) for w, p in zip(words, target_probs)}

            results.append(
                {
                    "top_k": top_k_list,
                    "target_scores": target_scores,
                }
            )

    print(f"\nResults: {len(results)}")
    print(f'Prompt: "{prompt}"\n')
    # Print first 5 funny reviews
    funny_indices = sample[sample["label_is_funny_binary"] == 1].index[:2]
    print("=== FUNNY REVIEWS ===\n")
    for idx in funny_indices:
        i = sample.index.get_loc(idx)
        label_binary = (
            "Funny" if sample["label_is_funny_binary"].iloc[i] == 1 else "Not Funny"
        )
        label_minmax = sample["label_funny_minmax"].iloc[i]
        label_cat = sample["label_votes_funny_categorical"].iloc[i]
        print(
            f"--- Review {sample['review_id'].iloc[i]} Game: {sample['app_name'].iloc[i]} ---"
        )
        print(
            f"Funny Votes: {sample['review_votes_funny'].iloc[i]} --- label: [binary: {label_binary}, norm: {label_minmax}, cat: {label_cat}]"
        )
        print(f"Text: {sample[column].iloc[i]}")
        print(f"Top-5: {results[i]['top_k']}")
        word_scores = sorted(
            ((w, results[i]["target_scores"][w]) for w in words),
            key=lambda x: x[1],
            reverse=True,
        )
        word_scores_str = ", ".join([f"{w}: {score}" for w, score in word_scores])
        print(f"Words:     {{{word_scores_str}}}")
        print()

    # Print first 5 not funny reviews
    not_funny_indices = sample[sample["label_is_funny_binary"] == 0].index[:2]
    print("\n=== NOT FUNNY REVIEWS ===\n")
    for idx in not_funny_indices:
        i = sample.index.get_loc(idx)
        label_binary = (
            "Funny" if sample["label_is_funny_binary"].iloc[i] == 1 else "Not Funny"
        )
        label_minmax = sample["label_funny_minmax"].iloc[i]
        label_cat = sample["label_votes_funny_categorical"].iloc[i]

        print(
            f"--- Review {sample['review_id'].iloc[i]} Game: {sample['app_name'].iloc[i]} ---"
        )
        print(
            f"Funny Votes: {sample['review_votes_funny'].iloc[i]} --- label: [binary: {label_binary}, norm: {label_minmax}, cat: {label_cat}]"
        )
        print(f"Text: {sample[column].iloc[i]}")
        print(f"Top-5: {results[i]['top_k']}")

        word_scores = sorted(
            ((w, results[i]["target_scores"][w]) for w in words),
            key=lambda x: x[1],
            reverse=True,
        )
        word_scores_str = ", ".join([f"{w}: {score}" for w, score in word_scores])
        print(f"Words:     {{{word_scores_str}}}")
        print()

    return results

## Spot checks with just review text

run the Spot checks for all prompts with just the review text as input

In [19]:
spot_check_reviews(
    sample=sample,
    column="review_text_cleaned",
    prompt=prompts_dezimal[0],
    tokenizer=tokenizer,
    model=model,
    device=device,
    words=cat_dezimal,
)

Prompt: "On a scale from 0.0 to 1.0 this review is 0.[MASK] funny."

=== HIGHEST funny votes ===
  Review ID : 144096803
  Game      : STAR WARS Jedi: Survivor
  Funny Votes: 1054  |  Label: Funny
  Text      : No one will read my review so I'll just say im gay
  Top-10: [('5', 0.262911), ('8', 0.120532), ('0', 0.112542), ('9', 0.093116), ('6', 0.092246), ('2', 0.057937), ('1', 0.055803), ('4', 0.033561), ('7', 0.030486), ('3', 0.023721)]
  Words     : {5: 0.262911, 8: 0.120532, 0: 0.112542, 9: 0.093116, 6: 0.092246, 2: 0.057937, 1: 0.055803, 4: 0.033561, 7: 0.030486, 3: 0.023721}
=== MEDIAN funny votes ===
  Review ID : 105581801
  Game      : ICARUS
  Funny Votes: 0  |  Label: Not Funny
  Text      : fuckin shite
  Top-10: [('5', 0.226135), ('9', 0.131668), ('8', 0.126682), ('6', 0.091547), ('0', 0.064057), ('1', 0.048024), ('2', 0.047188), ('7', 0.029822), ('4', 0.029256), ('3', 0.024649)]
  Words     : {5: 0.226135, 9: 0.131668, 8: 0.126682, 6: 0.091547, 0: 0.064057, 1: 0.048024, 2

In [20]:
spot_check_reviews(
    sample=sample,
    column="review_text_cleaned",
    prompt=prompts_dezimal[1],
    tokenizer=tokenizer,
    model=model,
    device=device,
    words=cat_dezimal,
)

Prompt: "On a scale from 0 to 1 this review is 0.[MASK] funny."

=== HIGHEST funny votes ===
  Review ID : 144096803
  Game      : STAR WARS Jedi: Survivor
  Funny Votes: 1054  |  Label: Funny
  Text      : No one will read my review so I'll just say im gay
  Top-10: [('5', 0.297825), ('8', 0.121303), ('6', 0.083852), ('9', 0.062794), ('2', 0.054472), ('25', 0.049385), ('4', 0.029726), ('1', 0.027551), ('10', 0.022508), ('7', 0.020046)]
  Words     : {5: 0.297825, 8: 0.121303, 6: 0.083852, 9: 0.062794, 2: 0.054472, 4: 0.029726, 1: 0.027551, 7: 0.020046, 3: 0.018181, 0: 0.006301}
=== MEDIAN funny votes ===
  Review ID : 105581801
  Game      : ICARUS
  Funny Votes: 0  |  Label: Not Funny
  Text      : fuckin shite
  Top-10: [('5', 0.286688), ('6', 0.078954), ('8', 0.074931), ('9', 0.072297), ('10', 0.049775), ('25', 0.047802), ('2', 0.031401), ('4', 0.02091), ('1', 0.019101), ('3', 0.015217)]
  Words     : {5: 0.286688, 6: 0.078954, 8: 0.074931, 9: 0.072297, 2: 0.031401, 4: 0.02091, 1: 

In [21]:
spot_check_reviews(
    sample=sample,
    column="review_text_cleaned",
    prompt=prompts_words[0],
    tokenizer=tokenizer,
    model=model,
    device=device,
    words=cat_words,
)

Prompt: "Overall, the humor is [MASK]."

=== HIGHEST funny votes ===
  Review ID : 144096803
  Game      : STAR WARS Jedi: Survivor
  Funny Votes: 1054  |  Label: Funny
  Text      : No one will read my review so I'll just say im gay
  Top-10: [('good', 0.163697), ('infectious', 0.080627), ('great', 0.08011), ('fantastic', 0.03025), ('excellent', 0.028206), ('amazing', 0.02215), ('there', 0.018352), ('funny', 0.016814), ('refreshing', 0.01597), ('hilarious', 0.015327)]
  Words     : {hilarious: 0.015327, amusing: 0.002945, hysterical: 0.000537, witty: 0.000495, serious: 0.000471}
=== MEDIAN funny votes ===
  Review ID : 105581801
  Game      : ICARUS
  Funny Votes: 0  |  Label: Not Funny
  Text      : fuckin shite
  Top-10: [('good', 0.258912), ('great', 0.043027), ('infectious', 0.030121), ('there', 0.029279), ('bad', 0.0258), ('excellent', 0.023948), ('right', 0.018721), ('hilarious', 0.012833), ('funny', 0.012035), ('gone', 0.009755)]
  Words     : {hilarious: 0.012833, amusing: 0.0

In [22]:
spot_check_reviews(
    sample=sample,
    column="review_text_cleaned",
    prompt=prompts_words[1],
    tokenizer=tokenizer,
    model=model,
    device=device,
    words=cat_words,
)

Prompt: "Overall the humor of this review is [MASK]."

=== HIGHEST funny votes ===
  Review ID : 144096803
  Game      : STAR WARS Jedi: Survivor
  Funny Votes: 1054  |  Label: Funny
  Text      : No one will read my review so I'll just say im gay
  Top-10: [('great', 0.041647), ('simple', 0.040533), ('funny', 0.035945), ('amazing', 0.034154), ('good', 0.032693), ('interesting', 0.031705), ('infectious', 0.021129), ('fantastic', 0.020226), ('hilarious', 0.019483), ('excellent', 0.018368)]
  Words     : {hilarious: 0.019483, amusing: 0.008932, serious: 0.002069, witty: 0.001261, hysterical: 0.00081}
=== MEDIAN funny votes ===
  Review ID : 105581801
  Game      : ICARUS
  Funny Votes: 0  |  Label: Not Funny
  Text      : fuckin shite
  Top-10: [('good', 0.046459), ('great', 0.04014), ('excellent', 0.031646), ('bad', 0.022712), ('crude', 0.020621), ('hilarious', 0.017409), ('funny', 0.016541), ('infectious', 0.014261), ('terrible', 0.012987), ('interesting', 0.01251)]
  Words     : {hila

In [23]:
spot_check_reviews(
    sample=sample,
    column="review_text_cleaned",
    prompt=prompts_words[2],
    tokenizer=tokenizer,
    model=model,
    device=device,
    words=cat_words,
)

Prompt: "The humor in this review is [MASK]."

=== HIGHEST funny votes ===
  Review ID : 144096803
  Game      : STAR WARS Jedi: Survivor
  Funny Votes: 1054  |  Label: Funny
  Text      : No one will read my review so I'll just say im gay
  Top-10: [('good', 0.057474), ('funny', 0.053435), ('bad', 0.047501), ('great', 0.03495), ('hilarious', 0.03481), ('ridiculous', 0.026522), ('amazing', 0.021823), ('awful', 0.019384), ('fantastic', 0.018015), ('excellent', 0.017612)]
  Words     : {hilarious: 0.03481, amusing: 0.007094, hysterical: 0.001415, serious: 0.001233, witty: 0.001002}
=== MEDIAN funny votes ===
  Review ID : 105581801
  Game      : ICARUS
  Funny Votes: 0  |  Label: Not Funny
  Text      : fuckin shite
  Top-10: [('good', 0.068369), ('bad', 0.053023), ('great', 0.033241), ('hilarious', 0.027936), ('awful', 0.025159), ('excellent', 0.023727), ('funny', 0.020197), ('infectious', 0.017878), ('terrible', 0.016979), ('ridiculous', 0.016347)]
  Words     : {hilarious: 0.027936, a

## Run mask Predictions for whol sample for just review text

In [24]:
result = {}

In [27]:
result["dezimal0"] = get_mask_predictions(
    sample=sample,
    column="review_text_cleaned",
    words=cat_dezimal,
    prompt=prompts_dezimal[0],
    tokenizer=tokenizer,
    model=model,
    device=device,
)

Predicting (batched):   0%|          | 0/305 [00:00<?, ?it/s]


Results: 9760
Prompt: "On a scale from 0.0 to 1.0 this review is 0.[MASK] funny."

=== FUNNY REVIEWS ===

--- Review 174289382 Game: Brotato ---
Funny Votes: 1 --- label: [binary: Funny, norm: 0.0002781641168289, cat: 1]
Text: dont ask me why i play this so much for i do not know, but its a lot of fun with no strings attached^^
Top-5: [('5', 0.225711), ('0', 0.137489), ('9', 0.109047), ('8', 0.094476), ('6', 0.082008), ('1', 0.048878), ('2', 0.04873), ('4', 0.033848), ('7', 0.030155), ('3', 0.024706)]
Words:     {5: 0.225711, 0: 0.137489, 9: 0.109047, 8: 0.094476, 6: 0.082008, 1: 0.048878, 2: 0.04873, 4: 0.033848, 7: 0.030155, 3: 0.024706}

--- Review 182835611 Game: Factorio ---
Funny Votes: 4 --- label: [binary: Funny, norm: 0.0011126564673157, cat: 2]
Text: it looks like a lot of fun in streams. but when i played it myself, it felt like such a chore. 
vanilla factorio was already a lot of work to get to the end. but now? you need to handle so many new concepts in parallel. its exha

In [ ]:
result["dezimal1"] = get_mask_predictions(
    sample=sample,
    column="review_text_cleaned",
    words=cat_dezimal,
    prompt=prompts_dezimal[1],
    tokenizer=tokenizer,
    model=model,
    device=device,
)

In [ ]:
result["words0"] = get_mask_predictions(
    sample=sample,
    column="review_text_cleaned",
    words=cat_words,
    prompt=prompts_words[0],
    tokenizer=tokenizer,
    model=model,
    device=device,
)

In [ ]:
result["words1"] = get_mask_predictions(
    sample=sample,
    column="review_text_cleaned",
    words=cat_words,
    prompt=prompts_words[1],
    tokenizer=tokenizer,
    model=model,
    device=device,
)

In [ ]:
result["words2"] = get_mask_predictions(
    sample=sample,
    column="review_text_cleaned",
    words=cat_words,
    prompt=prompts_words[2],
    tokenizer=tokenizer,
    model=model,
    device=device,
)

# Spot checks with review + app meta data

run spot checks with review + app meta data as input

In [ ]:
spot_check_reviews(
    sample=sample,
    column="review_info_text_cleaned",
    prompt=prompts_dezimal[0],
    tokenizer=tokenizer,
    model=model,
    device=device,
    words=cat_dezimal,
)

In [ ]:
spot_check_reviews(
    sample=sample,
    column="review_info_text_cleaned",
    prompt=prompts_dezimal[1],
    tokenizer=tokenizer,
    model=model,
    device=device,
    words=cat_dezimal,
)

In [ ]:
spot_check_reviews(
    sample=sample,
    column="review_info_text_cleaned",
    prompt=prompts_words[0],
    tokenizer=tokenizer,
    model=model,
    device=device,
    words=cat_words,
)

In [ ]:
spot_check_reviews(
    sample=sample,
    column="review_info_text_cleaned",
    prompt=prompts_words[1],
    tokenizer=tokenizer,
    model=model,
    device=device,
    words=cat_words,
)

In [ ]:
spot_check_reviews(
    sample=sample,
    column="review_info_text_cleaned",
    prompt=prompts_words[2],
    tokenizer=tokenizer,
    model=model,
    device=device,
    words=cat_words,
)

In [ ]:
result_info = {}

In [ ]:
result_info["dezimal0"] = get_mask_predictions(
    sample=sample,
    column="review_info_text_cleaned",
    words=cat_dezimal,
    prompt=prompts_dezimal[0],
    tokenizer=tokenizer,
    model=model,
    device=device,
)

In [ ]:
result_info["dezimal1"] = get_mask_predictions(
    sample=sample,
    column="review_info_text_cleaned",
    words=cat_dezimal,
    prompt=prompts_dezimal[1],
    tokenizer=tokenizer,
    model=model,
    device=device,
)

In [ ]:
result_info["words0"] = get_mask_predictions(
    sample=sample,
    column="review_info_text_cleaned",
    words=cat_words,
    prompt=prompts_words[0],
    tokenizer=tokenizer,
    model=model,
    device=device,
)

In [ ]:
result_info["words1"] = get_mask_predictions(
    sample=sample,
    column="review_info_text_cleaned",
    words=cat_words,
    prompt=prompts_words[1],
    tokenizer=tokenizer,
    model=model,
    device=device,
)

In [ ]:
result_info["words2"] = get_mask_predictions(
    sample=sample,
    column="review_info_text_cleaned",
    words=cat_words,
    prompt=prompts_words[2],
    tokenizer=tokenizer,
    model=model,
    device=device,
)